# Step 4 — Controlled Sensor-Degradation Framework

Objectives:

- reconstruct clean FD001 development/validation data from the frozen split
- preserve the original NASA raw files unchanged
- define controlled synthetic sensor degradation
- corrupt sensor channels only, not operating settings
- parameterize degradation using development-training statistics only
- reuse the frozen StandardScaler from Step 2
- keep corruption generation reproducible with fixed random seeds

Planned degradation types:
- additive noise
- bias
- gradual drift
- dropout / missingness
- stuck sensor

In [ ]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

# Locate project root
cwd = Path.cwd()

if (cwd / "02_dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "02_dataset").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

RAW_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "raw"
    / "CMAPSSData"
)

INTERIM_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "interim"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "processed"
    / "FD001"
)

print("Project root      :", PROJECT_ROOT)
print("Raw exists        :", RAW_DIR.exists())
print("Interim exists    :", INTERIM_DIR.exists())
print("Processed exists  :", PROCESSED_DIR.exists())

In [ ]:
with open(
    PROCESSED_DIR / "fd001_preprocessing_config.json",
    "r"
) as f:
    config = json.load(f)

scaler = joblib.load(
    PROCESSED_DIR / "fd001_standard_scaler.joblib"
)

split_df = pd.read_csv(
    INTERIM_DIR / "fd001_engine_split_seed42.csv"
)

PRIMARY_INPUT_COLUMNS = config[
    "primary_input_columns"
]

RETAINED_SENSOR_COLUMNS = [
    col
    for col in PRIMARY_INPUT_COLUMNS
    if col.startswith("sensor_")
]

RETAINED_OPERATING_COLUMNS = [
    col
    for col in PRIMARY_INPUT_COLUMNS
    if col.startswith("op_setting_")
]

print("Primary inputs          :", len(PRIMARY_INPUT_COLUMNS))
print("Retained sensors        :", len(RETAINED_SENSOR_COLUMNS))
print("Retained operating cols :", len(RETAINED_OPERATING_COLUMNS))

print("\nSensors:")
print(RETAINED_SENSOR_COLUMNS)

print("\nOperating settings:")
print(RETAINED_OPERATING_COLUMNS)

In [ ]:
column_names = (
    ["unit_id", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

fd001_raw = pd.read_csv(
    RAW_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

print("Raw FD001 shape:", fd001_raw.shape)
print("Raw engines     :", fd001_raw["unit_id"].nunique())

In [ ]:
dev_engine_ids = set(
    split_df.loc[
        split_df["split"] == "dev_train",
        "unit_id"
    ]
)

val_engine_ids = set(
    split_df.loc[
        split_df["split"] == "validation",
        "unit_id"
    ]
)

dev_raw = (
    fd001_raw[
        fd001_raw["unit_id"].isin(dev_engine_ids)
    ]
    .sort_values(["unit_id", "cycle"])
    .reset_index(drop=True)
)

val_raw = (
    fd001_raw[
        fd001_raw["unit_id"].isin(val_engine_ids)
    ]
    .sort_values(["unit_id", "cycle"])
    .reset_index(drop=True)
)

print("Development rows   :", len(dev_raw))
print("Validation rows    :", len(val_raw))

print("Development engines:", dev_raw["unit_id"].nunique())
print("Validation engines :", val_raw["unit_id"].nunique())

print(
    "Engine overlap:",
    set(dev_raw["unit_id"]).intersection(
        set(val_raw["unit_id"])
    )
)

In [ ]:
def add_rul_targets(df):
    df = df.copy()

    max_cycle = (
        df.groupby("unit_id")["cycle"]
        .transform("max")
    )

    df["RUL_raw"] = (
        max_cycle - df["cycle"]
    )

    df["RUL_cap_125"] = np.minimum(
        df["RUL_raw"],
        125
    )

    df["RUL_cap_130"] = np.minimum(
        df["RUL_raw"],
        130
    )

    return df


dev_raw = add_rul_targets(dev_raw)
val_raw = add_rul_targets(val_raw)

print("RUL targets reconstructed.")

In [ ]:
saved_dev = pd.read_csv(
    PROCESSED_DIR / "fd001_dev_train_scaled.csv"
)

saved_val = pd.read_csv(
    PROCESSED_DIR / "fd001_validation_scaled.csv"
)

In [ ]:
dev_clean_scaled = dev_raw.copy()
val_clean_scaled = val_raw.copy()

dev_clean_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    dev_raw[PRIMARY_INPUT_COLUMNS]
)

val_clean_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    val_raw[PRIMARY_INPUT_COLUMNS]
)

In [ ]:
assert len(dev_raw) == 16561
assert len(val_raw) == 4070

assert dev_raw["unit_id"].nunique() == 80
assert val_raw["unit_id"].nunique() == 20

assert np.array_equal(
    dev_clean_scaled["unit_id"].to_numpy(),
    saved_dev["unit_id"].to_numpy()
)

assert np.array_equal(
    val_clean_scaled["unit_id"].to_numpy(),
    saved_val["unit_id"].to_numpy()
)

assert np.array_equal(
    dev_clean_scaled["cycle"].to_numpy(),
    saved_dev["cycle"].to_numpy()
)

assert np.array_equal(
    val_clean_scaled["cycle"].to_numpy(),
    saved_val["cycle"].to_numpy()
)

assert np.allclose(
    dev_clean_scaled[
        PRIMARY_INPUT_COLUMNS
    ].to_numpy(),
    saved_dev[
        PRIMARY_INPUT_COLUMNS
    ].to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

assert np.allclose(
    val_clean_scaled[
        PRIMARY_INPUT_COLUMNS
    ].to_numpy(),
    saved_val[
        PRIMARY_INPUT_COLUMNS
    ].to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

print("Raw-data reconstruction reproduces Step 2 exactly.")

In [ ]:
sensor_reference_stats = pd.DataFrame({
    "mean": dev_raw[RETAINED_SENSOR_COLUMNS].mean(),
    "std": dev_raw[RETAINED_SENSOR_COLUMNS].std(ddof=0),
    "min": dev_raw[RETAINED_SENSOR_COLUMNS].min(),
    "max": dev_raw[RETAINED_SENSOR_COLUMNS].max(),
    "q01": dev_raw[RETAINED_SENSOR_COLUMNS].quantile(0.01),
    "q99": dev_raw[RETAINED_SENSOR_COLUMNS].quantile(0.99),
})

sensor_reference_stats["range"] = (
    sensor_reference_stats["max"]
    - sensor_reference_stats["min"]
)

display(
    sensor_reference_stats.round(6)
)

In [ ]:
scaler_reference = pd.DataFrame({
    "feature": PRIMARY_INPUT_COLUMNS,
    "scaler_mean": scaler.mean_,
    "scaler_scale": scaler.scale_
}).set_index("feature")

sensor_scaler_reference = scaler_reference.loc[
    RETAINED_SENSOR_COLUMNS
]

assert np.allclose(
    sensor_reference_stats["mean"].to_numpy(),
    sensor_scaler_reference["scaler_mean"].to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

assert np.allclose(
    sensor_reference_stats["std"].to_numpy(),
    sensor_scaler_reference["scaler_scale"].to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

print(
    "Sensor reference statistics match the frozen scaler."
)

In [ ]:
sensor_diff_records = []

for sensor in RETAINED_SENSOR_COLUMNS:

    diffs = (
        dev_raw
        .groupby("unit_id")[sensor]
        .diff()
        .dropna()
    )

    sensor_diff_records.append({
        "sensor": sensor,
        "diff_std": diffs.std(ddof=0),
        "median_abs_diff": diffs.abs().median(),
        "q95_abs_diff": diffs.abs().quantile(0.95)
    })


sensor_temporal_stats = (
    pd.DataFrame(sensor_diff_records)
    .set_index("sensor")
)

display(
    sensor_temporal_stats.round(6)
)

In [ ]:
CORRUPTION_SEED = 42

CORRUPTION_LEVELS = {
    "noise": {
        "low": 0.10,
        "medium": 0.25,
        "high": 0.50,
    },

    "bias": {
        "low": 0.25,
        "medium": 0.50,
        "high": 1.00,
    },

    "drift": {
        "low": 0.25,
        "medium": 0.50,
        "high": 1.00,
    },

    "dropout": {
        "low": 0.10,
        "medium": 0.25,
        "high": 0.50,
    },

    "stuck": {
        "low": 0.20,
        "medium": 0.40,
        "high": 0.60,
    }
}

print("Corruption seed:", CORRUPTION_SEED)

for corruption, levels in CORRUPTION_LEVELS.items():
    print(corruption, ":", levels)

In [ ]:
DEGRADATION_DIR = (
    PROJECT_ROOT
    / "05_experiments"
    / "sensor_degradation"
)

DEGRADATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

sensor_reference_stats.to_csv(
    DEGRADATION_DIR / "fd001_sensor_reference_stats.csv"
)

sensor_temporal_stats.to_csv(
    DEGRADATION_DIR / "fd001_sensor_temporal_stats.csv"
)

print("Sensor reference statistics saved.")

In [ ]:
def add_sensor_noise(
    df,
    sensor,
    severity,
    sensor_stats,
    seed=42
):
    corrupted = df.copy()

    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    sensor_std = sensor_stats.loc[
        sensor,
        "std"
    ]

    noise_std = severity * sensor_std

    rng = np.random.default_rng(seed)

    noise = rng.normal(
        loc=0.0,
        scale=noise_std,
        size=len(corrupted)
    )

    corrupted[sensor] = (
        corrupted[sensor].to_numpy()
        + noise
    )

    return corrupted, noise

In [ ]:
TEST_SENSOR = "sensor_2"

TEST_SEVERITY_NAME = "medium"

TEST_ALPHA = CORRUPTION_LEVELS[
    "noise"
][TEST_SEVERITY_NAME]

val_noise_test, injected_noise = (
    add_sensor_noise(
        df=val_raw,
        sensor=TEST_SENSOR,
        severity=TEST_ALPHA,
        sensor_stats=sensor_reference_stats,
        seed=CORRUPTION_SEED
    )
)

print("Sensor       :", TEST_SENSOR)
print("Severity     :", TEST_SEVERITY_NAME)
print("Alpha        :", TEST_ALPHA)

print(
    "Reference std:",
    sensor_reference_stats.loc[
        TEST_SENSOR,
        "std"
    ]
)

print(
    "Target noise std:",
    TEST_ALPHA
    * sensor_reference_stats.loc[
        TEST_SENSOR,
        "std"
    ]
)

print(
    "Actual noise std:",
    injected_noise.std(ddof=0)
)

print(
    "Actual noise mean:",
    injected_noise.mean()
)

In [ ]:
# The corrupted sensor should change
assert not np.allclose(
    val_noise_test[TEST_SENSOR],
    val_raw[TEST_SENSOR]
)

# All other retained sensors must remain identical
other_sensors = [
    sensor
    for sensor in RETAINED_SENSOR_COLUMNS
    if sensor != TEST_SENSOR
]

assert np.allclose(
    val_noise_test[other_sensors].to_numpy(),
    val_raw[other_sensors].to_numpy()
)

# Operating settings must remain clean
assert np.allclose(
    val_noise_test[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy(),
    val_raw[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy()
)

# Metadata and RUL targets must not change
for col in [
    "unit_id",
    "cycle",
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130"
]:
    assert np.array_equal(
        val_noise_test[col].to_numpy(),
        val_raw[col].to_numpy()
    )

print(
    "Single-sensor noise integrity checks passed."
)

In [ ]:
val_noise_repeat, noise_repeat = (
    add_sensor_noise(
        df=val_raw,
        sensor=TEST_SENSOR,
        severity=TEST_ALPHA,
        sensor_stats=sensor_reference_stats,
        seed=CORRUPTION_SEED
    )
)

assert np.array_equal(
    injected_noise,
    noise_repeat
)

assert np.array_equal(
    val_noise_test[TEST_SENSOR].to_numpy(),
    val_noise_repeat[TEST_SENSOR].to_numpy()
)

print(
    "Noise corruption is reproducible with the fixed seed."
)

In [ ]:
val_noise_scaled = val_noise_test.copy()

val_noise_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    val_noise_test[
        PRIMARY_INPUT_COLUMNS
    ]
)

In [ ]:
scaled_difference = (
    val_noise_scaled[TEST_SENSOR]
    - val_clean_scaled[TEST_SENSOR]
)

print(
    "Scaled perturbation mean:",
    scaled_difference.mean()
)

print(
    "Scaled perturbation std:",
    scaled_difference.std(ddof=0)
)

In [ ]:
def make_noise_scenario(
    clean_df,
    sensor,
    severity_name,
    seed=42
):
    if severity_name not in CORRUPTION_LEVELS["noise"]:
        raise ValueError(
            f"Unknown noise severity: {severity_name}"
        )

    alpha = CORRUPTION_LEVELS[
        "noise"
    ][severity_name]

    corrupted_df, noise = add_sensor_noise(
        df=clean_df,
        sensor=sensor,
        severity=alpha,
        sensor_stats=sensor_reference_stats,
        seed=seed
    )

    return corrupted_df, noise

In [ ]:
noise_validation_records = []

for sensor in RETAINED_SENSOR_COLUMNS:

    reference_std = sensor_reference_stats.loc[
        sensor,
        "std"
    ]

    for severity_name, alpha in (
        CORRUPTION_LEVELS["noise"].items()
    ):

        corrupted_df, noise = make_noise_scenario(
            clean_df=val_raw,
            sensor=sensor,
            severity_name=severity_name,
            seed=CORRUPTION_SEED
        )

        target_noise_std = (
            alpha * reference_std
        )

        actual_noise_std = (
            noise.std(ddof=0)
        )

        noise_validation_records.append({
            "sensor": sensor,
            "severity": severity_name,
            "alpha": alpha,
            "target_noise_std": target_noise_std,
            "actual_noise_std": actual_noise_std,
            "ratio_actual_to_target":
                actual_noise_std
                / target_noise_std
        })


noise_validation_df = pd.DataFrame(
    noise_validation_records
)

display(
    noise_validation_df.round(6)
)

In [ ]:
print(
    "Number of validated noise scenarios:",
    len(noise_validation_df)
)

print(
    "Minimum actual/target ratio:",
    noise_validation_df[
        "ratio_actual_to_target"
    ].min()
)

print(
    "Maximum actual/target ratio:",
    noise_validation_df[
        "ratio_actual_to_target"
    ].max()
)

assert len(noise_validation_df) == 45

assert (
    noise_validation_df[
        "ratio_actual_to_target"
    ].between(0.90, 1.10)
    .all()
)

print(
    "All 45 additive-noise scenarios passed."
)

In [ ]:
def add_sensor_bias(
    df,
    sensor,
    severity,
    direction,
    sensor_stats
):
    corrupted = df.copy()

    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be either -1 or +1."
        )

    sensor_std = sensor_stats.loc[
        sensor,
        "std"
    ]

    bias_amount = (
        direction
        * severity
        * sensor_std
    )

    corrupted[sensor] = (
        corrupted[sensor]
        + bias_amount
    )

    return corrupted, bias_amount

In [ ]:
BIAS_TEST_SENSOR = "sensor_2"
BIAS_TEST_SEVERITY = "medium"
BIAS_TEST_DIRECTION = 1

BIAS_ALPHA = CORRUPTION_LEVELS[
    "bias"
][BIAS_TEST_SEVERITY]

val_bias_test, bias_amount = (
    add_sensor_bias(
        df=val_raw,
        sensor=BIAS_TEST_SENSOR,
        severity=BIAS_ALPHA,
        direction=BIAS_TEST_DIRECTION,
        sensor_stats=sensor_reference_stats
    )
)

print("Sensor       :", BIAS_TEST_SENSOR)
print("Severity     :", BIAS_TEST_SEVERITY)
print("Alpha        :", BIAS_ALPHA)
print("Direction    :", BIAS_TEST_DIRECTION)
print("Bias amount  :", bias_amount)

In [ ]:
observed_shift = (
    val_bias_test[BIAS_TEST_SENSOR]
    - val_raw[BIAS_TEST_SENSOR]
)

print(
    "Observed mean shift:",
    observed_shift.mean()
)

print(
    "Observed shift std:",
    observed_shift.std(ddof=0)
)

assert np.allclose(
    observed_shift.to_numpy(),
    bias_amount
)

other_sensors = [
    sensor
    for sensor in RETAINED_SENSOR_COLUMNS
    if sensor != BIAS_TEST_SENSOR
]

assert np.allclose(
    val_bias_test[other_sensors].to_numpy(),
    val_raw[other_sensors].to_numpy()
)

assert np.allclose(
    val_bias_test[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy(),
    val_raw[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy()
)

for col in [
    "unit_id",
    "cycle",
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130"
]:
    assert np.array_equal(
        val_bias_test[col].to_numpy(),
        val_raw[col].to_numpy()
    )

print("Single-sensor bias integrity checks passed.")

In [ ]:
val_bias_scaled = val_bias_test.copy()

val_bias_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    val_bias_test[
        PRIMARY_INPUT_COLUMNS
    ]
)

scaled_bias_shift = (
    val_bias_scaled[BIAS_TEST_SENSOR]
    - val_clean_scaled[BIAS_TEST_SENSOR]
)

print(
    "Scaled bias mean:",
    scaled_bias_shift.mean()
)

print(
    "Scaled bias std:",
    scaled_bias_shift.std(ddof=0)
)

In [ ]:
bias_validation_records = []

for sensor in RETAINED_SENSOR_COLUMNS:

    reference_std = sensor_reference_stats.loc[
        sensor,
        "std"
    ]

    for severity_name, alpha in (
        CORRUPTION_LEVELS["bias"].items()
    ):

        for direction in (-1, 1):

            corrupted_df, bias_amount = (
                add_sensor_bias(
                    df=val_raw,
                    sensor=sensor,
                    severity=alpha,
                    direction=direction,
                    sensor_stats=sensor_reference_stats
                )
            )

            observed_shift = (
                corrupted_df[sensor]
                - val_raw[sensor]
            )

            expected_shift = (
                direction
                * alpha
                * reference_std
            )

            bias_validation_records.append({
                "sensor": sensor,
                "severity": severity_name,
                "alpha": alpha,
                "direction": direction,
                "expected_shift": expected_shift,
                "observed_mean_shift":
                    observed_shift.mean(),
                "observed_shift_std":
                    observed_shift.std(ddof=0)
            })


bias_validation_df = pd.DataFrame(
    bias_validation_records
)

display(
    bias_validation_df.round(6)
)

In [ ]:
print(
    "Number of validated bias scenarios:",
    len(bias_validation_df)
)

assert len(bias_validation_df) == 90

assert np.allclose(
    bias_validation_df["expected_shift"],
    bias_validation_df["observed_mean_shift"],
    rtol=1e-10,
    atol=1e-12
)

assert np.allclose(
    bias_validation_df["observed_shift_std"],
    0.0,
    atol=1e-12
)

print(
    "All 90 constant-bias scenarios passed."
)

In [ ]:
def add_sensor_drift(
    df,
    sensor,
    severity,
    direction,
    sensor_stats
):
    corrupted = df.copy()

    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be either -1 or +1."
        )

    sensor_std = sensor_stats.loc[
        sensor,
        "std"
    ]

    final_drift = (
        direction
        * severity
        * sensor_std
    )

    drift_values = np.zeros(
        len(corrupted),
        dtype=float
    )

    for unit_id, group in corrupted.groupby(
        "unit_id",
        sort=False
    ):

        group = group.sort_values("cycle")

        indices = group.index.to_numpy()

        progress = np.linspace(
            0.0,
            1.0,
            len(group)
        )

        drift_values[indices] = (
            final_drift * progress
        )

    corrupted[sensor] = (
        corrupted[sensor].to_numpy()
        + drift_values
    )

    return (
        corrupted,
        drift_values,
        final_drift
    )

In [ ]:
DRIFT_TEST_SENSOR = "sensor_2"
DRIFT_TEST_SEVERITY = "medium"
DRIFT_TEST_DIRECTION = 1

DRIFT_ALPHA = CORRUPTION_LEVELS[
    "drift"
][DRIFT_TEST_SEVERITY]

val_drift_test, injected_drift, final_drift = (
    add_sensor_drift(
        df=val_raw,
        sensor=DRIFT_TEST_SENSOR,
        severity=DRIFT_ALPHA,
        direction=DRIFT_TEST_DIRECTION,
        sensor_stats=sensor_reference_stats
    )
)

print("Sensor              :", DRIFT_TEST_SENSOR)
print("Severity            :", DRIFT_TEST_SEVERITY)
print("Alpha               :", DRIFT_ALPHA)
print("Direction           :", DRIFT_TEST_DIRECTION)
print("Final drift amount  :", final_drift)

In [ ]:
drift_check_df = val_drift_test[
    ["unit_id", "cycle"]
].copy()

drift_check_df["drift"] = injected_drift

first_drift = (
    drift_check_df
    .groupby("unit_id")["drift"]
    .first()
)

last_drift = (
    drift_check_df
    .groupby("unit_id")["drift"]
    .last()
)

print(
    "Largest absolute starting drift:",
    first_drift.abs().max()
)

print(
    "Smallest final drift:",
    last_drift.min()
)

print(
    "Largest final drift:",
    last_drift.max()
)

assert np.allclose(
    first_drift.to_numpy(),
    0.0,
    atol=1e-12
)

assert np.allclose(
    last_drift.to_numpy(),
    final_drift,
    rtol=1e-10,
    atol=1e-12
)

print(
    "Per-engine drift endpoint checks passed."
)

In [ ]:
other_sensors = [
    sensor
    for sensor in RETAINED_SENSOR_COLUMNS
    if sensor != DRIFT_TEST_SENSOR
]

assert np.allclose(
    val_drift_test[other_sensors].to_numpy(),
    val_raw[other_sensors].to_numpy()
)

assert np.allclose(
    val_drift_test[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy(),
    val_raw[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy()
)

for col in [
    "unit_id",
    "cycle",
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130"
]:
    assert np.array_equal(
        val_drift_test[col].to_numpy(),
        val_raw[col].to_numpy()
    )

print(
    "Single-sensor drift integrity checks passed."
)

In [ ]:
val_drift_scaled = val_drift_test.copy()

val_drift_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    val_drift_test[
        PRIMARY_INPUT_COLUMNS
    ]
)

scaled_drift = (
    val_drift_scaled[DRIFT_TEST_SENSOR]
    - val_clean_scaled[DRIFT_TEST_SENSOR]
)

scaled_drift_check = pd.DataFrame({
    "unit_id": val_raw["unit_id"],
    "cycle": val_raw["cycle"],
    "scaled_drift": scaled_drift
})

scaled_first = (
    scaled_drift_check
    .groupby("unit_id")["scaled_drift"]
    .first()
)

scaled_last = (
    scaled_drift_check
    .groupby("unit_id")["scaled_drift"]
    .last()
)

print(
    "Maximum absolute scaled starting drift:",
    scaled_first.abs().max()
)

print(
    "Mean scaled final drift:",
    scaled_last.mean()
)

In [ ]:
def add_sensor_drift(
    df,
    sensor,
    severity,
    direction,
    sensor_stats
):
    corrupted = df.copy()

    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be either -1 or +1."
        )

    sensor_std = sensor_stats.loc[
        sensor,
        "std"
    ]

    final_drift = (
        direction
        * severity
        * sensor_std
    )

    # Index-safe drift storage
    drift_values = pd.Series(
        0.0,
        index=corrupted.index,
        dtype=float
    )

    for unit_id, group in corrupted.groupby(
        "unit_id",
        sort=False
    ):

        group = group.sort_values("cycle")

        progress = np.linspace(
            0.0,
            1.0,
            len(group)
        )

        drift_values.loc[group.index] = (
            final_drift * progress
        )

    corrupted[sensor] = (
        corrupted[sensor]
        + drift_values
    )

    return (
        corrupted,
        drift_values.to_numpy(),
        final_drift
    )

In [ ]:
drift_validation_records = []

for sensor in RETAINED_SENSOR_COLUMNS:

    reference_std = sensor_reference_stats.loc[
        sensor,
        "std"
    ]

    for severity_name, alpha in (
        CORRUPTION_LEVELS["drift"].items()
    ):

        for direction in (-1, 1):

            corrupted_df, drift_values, final_drift = (
                add_sensor_drift(
                    df=val_raw,
                    sensor=sensor,
                    severity=alpha,
                    direction=direction,
                    sensor_stats=sensor_reference_stats
                )
            )

            drift_check = pd.DataFrame({
                "unit_id": val_raw["unit_id"].to_numpy(),
                "cycle": val_raw["cycle"].to_numpy(),
                "drift": drift_values
            })

            first_values = (
                drift_check
                .groupby("unit_id")["drift"]
                .first()
            )

            last_values = (
                drift_check
                .groupby("unit_id")["drift"]
                .last()
            )

            expected_final = (
                direction
                * alpha
                * reference_std
            )

            drift_validation_records.append({
                "sensor": sensor,
                "severity": severity_name,
                "alpha": alpha,
                "direction": direction,
                "expected_final_drift":
                    expected_final,
                "max_abs_starting_drift":
                    first_values.abs().max(),
                "mean_final_drift":
                    last_values.mean(),
                "min_final_drift":
                    last_values.min(),
                "max_final_drift":
                    last_values.max()
            })


drift_validation_df = pd.DataFrame(
    drift_validation_records
)

display(
    drift_validation_df.round(6)
)

In [ ]:
print(
    "Number of validated drift scenarios:",
    len(drift_validation_df)
)

assert len(drift_validation_df) == 90

# Every engine must begin with zero injected drift
assert np.allclose(
    drift_validation_df[
        "max_abs_starting_drift"
    ],
    0.0,
    atol=1e-12
)

# Final drift must equal the requested value
assert np.allclose(
    drift_validation_df[
        "expected_final_drift"
    ],
    drift_validation_df[
        "mean_final_drift"
    ],
    rtol=1e-10,
    atol=1e-12
)

assert np.allclose(
    drift_validation_df[
        "expected_final_drift"
    ],
    drift_validation_df[
        "min_final_drift"
    ],
    rtol=1e-10,
    atol=1e-12
)

assert np.allclose(
    drift_validation_df[
        "expected_final_drift"
    ],
    drift_validation_df[
        "max_final_drift"
    ],
    rtol=1e-10,
    atol=1e-12
)

print(
    "All 90 gradual-drift scenarios passed."
)

In [ ]:
def add_sensor_dropout(
    df,
    sensor,
    dropout_rate,
    sensor_stats,
    seed=42
):
    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    if not 0.0 <= dropout_rate < 1.0:
        raise ValueError(
            "dropout_rate must be between 0 and 1."
        )

    missing_df = df.copy()

    n_rows = len(missing_df)

    n_drop = int(
        round(dropout_rate * n_rows)
    )

    rng = np.random.default_rng(seed)

    drop_positions = rng.choice(
        n_rows,
        size=n_drop,
        replace=False
    )

    drop_mask = np.zeros(
        n_rows,
        dtype=bool
    )

    drop_mask[drop_positions] = True

    drop_index = missing_df.index[
        drop_positions
    ]

    # Introduce actual missing values
    missing_df.loc[
        drop_index,
        sensor
    ] = np.nan

    # Leakage-safe imputation
    impute_value = sensor_stats.loc[
        sensor,
        "mean"
    ]

    imputed_df = missing_df.copy()

    imputed_df[sensor] = (
        imputed_df[sensor]
        .fillna(impute_value)
    )

    return (
        missing_df,
        imputed_df,
        drop_mask,
        impute_value
    )

In [ ]:
DROPOUT_TEST_SENSOR = "sensor_2"
DROPOUT_TEST_SEVERITY = "medium"

DROPOUT_RATE = CORRUPTION_LEVELS[
    "dropout"
][DROPOUT_TEST_SEVERITY]

(
    val_dropout_missing,
    val_dropout_imputed,
    dropout_mask,
    dropout_impute_value
) = add_sensor_dropout(
    df=val_raw,
    sensor=DROPOUT_TEST_SENSOR,
    dropout_rate=DROPOUT_RATE,
    sensor_stats=sensor_reference_stats,
    seed=CORRUPTION_SEED
)

print("Sensor             :", DROPOUT_TEST_SENSOR)
print("Severity           :", DROPOUT_TEST_SEVERITY)
print("Requested fraction :", DROPOUT_RATE)

print(
    "Dropped readings   :",
    dropout_mask.sum()
)

print(
    "Actual fraction    :",
    dropout_mask.mean()
)

print(
    "Imputation value   :",
    dropout_impute_value
)

In [ ]:
print(
    "NaNs before imputation:",
    val_dropout_missing[
        DROPOUT_TEST_SENSOR
    ].isna().sum()
)

print(
    "NaNs after imputation:",
    val_dropout_imputed[
        DROPOUT_TEST_SENSOR
    ].isna().sum()
)

assert (
    val_dropout_missing[
        DROPOUT_TEST_SENSOR
    ].isna().sum()
    == dropout_mask.sum()
)

assert (
    val_dropout_imputed[
        DROPOUT_TEST_SENSOR
    ].isna().sum()
    == 0
)

assert np.allclose(
    val_dropout_imputed.loc[
        dropout_mask,
        DROPOUT_TEST_SENSOR
    ].to_numpy(),
    dropout_impute_value
)

print(
    "Dropout/imputation checks passed."
)

In [ ]:
other_sensors = [
    sensor
    for sensor in RETAINED_SENSOR_COLUMNS
    if sensor != DROPOUT_TEST_SENSOR
]

assert np.allclose(
    val_dropout_imputed[
        other_sensors
    ].to_numpy(),
    val_raw[
        other_sensors
    ].to_numpy()
)

assert np.allclose(
    val_dropout_imputed[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy(),
    val_raw[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy()
)

for col in [
    "unit_id",
    "cycle",
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130"
]:
    assert np.array_equal(
        val_dropout_imputed[col].to_numpy(),
        val_raw[col].to_numpy()
    )

print(
    "Single-sensor dropout integrity checks passed."
)

In [ ]:
val_dropout_scaled = (
    val_dropout_imputed.copy()
)

val_dropout_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    val_dropout_imputed[
        PRIMARY_INPUT_COLUMNS
    ]
)

In [ ]:
dropped_scaled_values = (
    val_dropout_scaled.loc[
        dropout_mask,
        DROPOUT_TEST_SENSOR
    ]
)

print(
    "Mean standardized value at dropped positions:",
    dropped_scaled_values.mean()
)

print(
    "Maximum absolute standardized value:",
    dropped_scaled_values.abs().max()
)

In [ ]:
def add_sensor_dropout(
    df,
    sensor,
    dropout_rate,
    sensor_stats,
    seed=42
):
    if sensor not in sensor_stats.index:
        raise ValueError(
            f"{sensor} not found in sensor reference statistics."
        )

    if not 0.0 <= dropout_rate < 1.0:
        raise ValueError(
            "dropout_rate must be between 0 and 1."
        )

    missing_df = df.copy()

    n_rows = len(missing_df)

    n_drop = int(
        round(dropout_rate * n_rows)
    )

    rng = np.random.default_rng(seed)

    # Same permutation for the same seed.
    # Taking prefixes guarantees nested
    # low < medium < high dropout masks.
    random_order = rng.permutation(
        n_rows
    )

    drop_positions = random_order[
        :n_drop
    ]

    drop_mask = np.zeros(
        n_rows,
        dtype=bool
    )

    drop_mask[drop_positions] = True

    drop_index = missing_df.index[
        drop_positions
    ]

    missing_df.loc[
        drop_index,
        sensor
    ] = np.nan

    impute_value = sensor_stats.loc[
        sensor,
        "mean"
    ]

    imputed_df = missing_df.copy()

    imputed_df[sensor] = (
        imputed_df[sensor]
        .fillna(impute_value)
    )

    return (
        missing_df,
        imputed_df,
        drop_mask,
        impute_value
    )

In [ ]:
dropout_validation_records = []

dropout_masks = {}

for sensor in RETAINED_SENSOR_COLUMNS:

    dropout_masks[sensor] = {}

    for severity_name, dropout_rate in (
        CORRUPTION_LEVELS["dropout"].items()
    ):

        (
            missing_df,
            imputed_df,
            mask,
            impute_value
        ) = add_sensor_dropout(
            df=val_raw,
            sensor=sensor,
            dropout_rate=dropout_rate,
            sensor_stats=sensor_reference_stats,
            seed=CORRUPTION_SEED
        )

        expected_count = int(
            round(
                dropout_rate
                * len(val_raw)
            )
        )

        assert mask.sum() == expected_count

        assert (
            missing_df[sensor]
            .isna()
            .sum()
            == expected_count
        )

        assert (
            imputed_df[sensor]
            .isna()
            .sum()
            == 0
        )

        assert np.allclose(
            imputed_df.loc[
                mask,
                sensor
            ].to_numpy(),
            impute_value
        )

        dropout_masks[
            sensor
        ][severity_name] = mask

        dropout_validation_records.append({
            "sensor": sensor,
            "severity": severity_name,
            "requested_fraction":
                dropout_rate,
            "expected_count":
                expected_count,
            "actual_count":
                int(mask.sum()),
            "actual_fraction":
                mask.mean(),
            "imputation_value":
                impute_value
        })


dropout_validation_df = pd.DataFrame(
    dropout_validation_records
)

display(
    dropout_validation_df.round(6)
)

In [ ]:
for sensor in RETAINED_SENSOR_COLUMNS:

    low_mask = dropout_masks[
        sensor
    ]["low"]

    medium_mask = dropout_masks[
        sensor
    ]["medium"]

    high_mask = dropout_masks[
        sensor
    ]["high"]

    # Every low-dropout point must also
    # belong to medium dropout.
    assert np.all(
        (~low_mask) | medium_mask
    )

    # Every medium-dropout point must also
    # belong to high dropout.
    assert np.all(
        (~medium_mask) | high_mask
    )


print(
    "Nested dropout-severity checks passed."
)

In [ ]:
print(
    "Number of validated dropout scenarios:",
    len(dropout_validation_df)
)

assert len(dropout_validation_df) == 45

assert np.array_equal(
    dropout_validation_df[
        "expected_count"
    ].to_numpy(),
    dropout_validation_df[
        "actual_count"
    ].to_numpy()
)

print(
    "All 45 dropout scenarios passed."
)

In [ ]:
def add_stuck_sensor(
    df,
    sensor,
    affected_fraction
):
    if sensor not in RETAINED_SENSOR_COLUMNS:
        raise ValueError(
            f"{sensor} is not a retained sensor."
        )

    if not 0.0 < affected_fraction < 1.0:
        raise ValueError(
            "affected_fraction must be between 0 and 1."
        )

    corrupted = df.copy()

    affected_mask = pd.Series(
        False,
        index=corrupted.index
    )

    stuck_records = []

    for unit_id, group in corrupted.groupby(
        "unit_id",
        sort=False
    ):

        group = group.sort_values("cycle")

        n_rows = len(group)

        n_affected = int(
            round(
                affected_fraction
                * n_rows
            )
        )

        # Ensure at least one affected point
        # and at least one valid point before
        # the sensor becomes stuck.
        n_affected = max(
            1,
            n_affected
        )

        n_affected = min(
            n_affected,
            n_rows - 1
        )

        onset_position = (
            n_rows - n_affected
        )

        indices = group.index.to_numpy()

        source_index = indices[
            onset_position - 1
        ]

        affected_indices = indices[
            onset_position:
        ]

        freeze_value = (
            corrupted.loc[
                source_index,
                sensor
            ]
        )

        corrupted.loc[
            affected_indices,
            sensor
        ] = freeze_value

        affected_mask.loc[
            affected_indices
        ] = True

        stuck_records.append({
            "unit_id": unit_id,
            "trajectory_length": n_rows,
            "requested_fraction":
                affected_fraction,
            "affected_count":
                n_affected,
            "actual_fraction":
                n_affected / n_rows,
            "source_cycle":
                group.iloc[
                    onset_position - 1
                ]["cycle"],
            "stuck_start_cycle":
                group.iloc[
                    onset_position
                ]["cycle"],
            "freeze_value":
                freeze_value
        })

    stuck_info = pd.DataFrame(
        stuck_records
    )

    return (
        corrupted,
        affected_mask.to_numpy(),
        stuck_info
    )

In [ ]:
STUCK_TEST_SENSOR = "sensor_2"
STUCK_TEST_SEVERITY = "medium"

STUCK_FRACTION = CORRUPTION_LEVELS[
    "stuck"
][STUCK_TEST_SEVERITY]

(
    val_stuck_test,
    stuck_mask,
    stuck_info
) = add_stuck_sensor(
    df=val_raw,
    sensor=STUCK_TEST_SENSOR,
    affected_fraction=STUCK_FRACTION
)

print("Sensor             :", STUCK_TEST_SENSOR)
print("Severity           :", STUCK_TEST_SEVERITY)
print("Requested fraction :", STUCK_FRACTION)

print(
    "Affected readings  :",
    stuck_mask.sum()
)

print(
    "Overall fraction   :",
    stuck_mask.mean()
)

display(
    stuck_info.head()
)

In [ ]:
for _, info in stuck_info.iterrows():

    unit_id = info["unit_id"]

    original_engine = (
        val_raw[
            val_raw["unit_id"] == unit_id
        ]
        .sort_values("cycle")
    )

    corrupted_engine = (
        val_stuck_test[
            val_stuck_test["unit_id"] == unit_id
        ]
        .sort_values("cycle")
    )

    n_affected = int(
        info["affected_count"]
    )

    onset_position = (
        len(original_engine)
        - n_affected
    )

    # Before the fault begins,
    # values must remain unchanged.
    assert np.allclose(
        corrupted_engine[
            STUCK_TEST_SENSOR
        ].iloc[
            :onset_position
        ].to_numpy(),

        original_engine[
            STUCK_TEST_SENSOR
        ].iloc[
            :onset_position
        ].to_numpy()
    )

    # Freeze value must be the
    # last valid reading before onset.
    expected_freeze_value = (
        original_engine[
            STUCK_TEST_SENSOR
        ].iloc[
            onset_position - 1
        ]
    )

    # Every value after onset must
    # remain stuck at that reading.
    assert np.allclose(
        corrupted_engine[
            STUCK_TEST_SENSOR
        ].iloc[
            onset_position:
        ].to_numpy(),

        expected_freeze_value
    )


print(
    "Per-engine stuck-sensor behavior checks passed."
)

In [ ]:
other_sensors = [
    sensor
    for sensor in RETAINED_SENSOR_COLUMNS
    if sensor != STUCK_TEST_SENSOR
]

assert np.allclose(
    val_stuck_test[
        other_sensors
    ].to_numpy(),
    val_raw[
        other_sensors
    ].to_numpy()
)

assert np.allclose(
    val_stuck_test[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy(),
    val_raw[
        RETAINED_OPERATING_COLUMNS
    ].to_numpy()
)

for col in [
    "unit_id",
    "cycle",
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130"
]:
    assert np.array_equal(
        val_stuck_test[
            col
        ].to_numpy(),
        val_raw[
            col
        ].to_numpy()
    )

print(
    "Single-sensor stuck integrity checks passed."
)

In [ ]:
stuck_validation_records = []
stuck_masks = {}

for sensor in RETAINED_SENSOR_COLUMNS:

    stuck_masks[sensor] = {}

    for severity_name, affected_fraction in (
        CORRUPTION_LEVELS["stuck"].items()
    ):

        (
            corrupted_df,
            affected_mask,
            scenario_info
        ) = add_stuck_sensor(
            df=val_raw,
            sensor=sensor,
            affected_fraction=affected_fraction
        )

        # -------------------------
        # Basic count checks
        # -------------------------
        expected_affected = int(
            scenario_info["affected_count"].sum()
        )

        actual_affected = int(
            affected_mask.sum()
        )

        assert actual_affected == expected_affected

        # -------------------------
        # Verify every engine
        # -------------------------
        for _, info in scenario_info.iterrows():

            unit_id = info["unit_id"]

            original_engine = (
                val_raw[
                    val_raw["unit_id"] == unit_id
                ]
                .sort_values("cycle")
            )

            corrupted_engine = (
                corrupted_df[
                    corrupted_df["unit_id"] == unit_id
                ]
                .sort_values("cycle")
            )

            n_affected = int(
                info["affected_count"]
            )

            onset_position = (
                len(original_engine)
                - n_affected
            )

            # Before onset must remain clean
            assert np.allclose(
                corrupted_engine[sensor]
                .iloc[:onset_position]
                .to_numpy(),

                original_engine[sensor]
                .iloc[:onset_position]
                .to_numpy()
            )

            # Last valid reading before failure
            freeze_value = (
                original_engine[sensor]
                .iloc[onset_position - 1]
            )

            # After onset all readings must be frozen
            assert np.allclose(
                corrupted_engine[sensor]
                .iloc[onset_position:]
                .to_numpy(),

                freeze_value
            )

        stuck_masks[
            sensor
        ][severity_name] = affected_mask

        stuck_validation_records.append({
            "sensor": sensor,
            "severity": severity_name,
            "requested_fraction":
                affected_fraction,
            "affected_count":
                actual_affected,
            "overall_fraction":
                affected_mask.mean(),
            "min_engine_fraction":
                scenario_info[
                    "actual_fraction"
                ].min(),
            "max_engine_fraction":
                scenario_info[
                    "actual_fraction"
                ].max()
        })


stuck_validation_df = pd.DataFrame(
    stuck_validation_records
)

display(
    stuck_validation_df.round(6)
)

In [ ]:
for sensor in RETAINED_SENSOR_COLUMNS:

    low_mask = stuck_masks[
        sensor
    ]["low"]

    medium_mask = stuck_masks[
        sensor
    ]["medium"]

    high_mask = stuck_masks[
        sensor
    ]["high"]

    assert np.all(
        (~low_mask) | medium_mask
    )

    assert np.all(
        (~medium_mask) | high_mask
    )


print(
    "Nested stuck-severity checks passed."
)

In [ ]:
print(
    "Number of validated stuck scenarios:",
    len(stuck_validation_df)
)

assert len(stuck_validation_df) == 45

assert (
    np.abs(
        stuck_validation_df[
            "overall_fraction"
        ]
        -
        stuck_validation_df[
            "requested_fraction"
        ]
    )
    < 0.005
).all()

print(
    "All 45 stuck-sensor scenarios passed."
)

In [ ]:
import json

corruption_spec = {
    "dataset": "NASA C-MAPSS FD001",
    "corruption_seed": CORRUPTION_SEED,

    "retained_sensors": RETAINED_SENSOR_COLUMNS,

    "levels": CORRUPTION_LEVELS,

    "noise": {
        "definition":
            "zero-mean Gaussian additive noise",
        "scale_basis":
            "development-training sensor population standard deviation",
        "directions":
            "not applicable"
    },

    "bias": {
        "definition":
            "constant additive sensor offset",
        "scale_basis":
            "development-training sensor population standard deviation",
        "directions":
            [-1, 1]
    },

    "drift": {
        "definition":
            "linear whole-trajectory drift from zero at first cycle "
            "to specified final magnitude at last observed cycle",
        "scale_basis":
            "development-training sensor population standard deviation",
        "directions":
            [-1, 1]
    },

    "dropout": {
        "definition":
            "random point missingness",
        "imputation":
            "development-training sensor mean",
        "nested_severity_masks":
            True
    },

    "stuck": {
        "definition":
            "stuck-at-last-valid-value for final fraction of each trajectory",
        "nested_severity_regions":
            True
    }
}

spec_path = (
    DEGRADATION_DIR
    / "fd001_corruption_specification.json"
)

with open(
    spec_path,
    "w"
) as f:
    json.dump(
        corruption_spec,
        f,
        indent=4
    )

print("Corruption specification saved to:")
print(spec_path)

In [ ]:
noise_validation_df.to_csv(
    DEGRADATION_DIR
    / "fd001_noise_validation.csv",
    index=False
)

bias_validation_df.to_csv(
    DEGRADATION_DIR
    / "fd001_bias_validation.csv",
    index=False
)

drift_validation_df.to_csv(
    DEGRADATION_DIR
    / "fd001_drift_validation.csv",
    index=False
)

dropout_validation_df.to_csv(
    DEGRADATION_DIR
    / "fd001_dropout_validation.csv",
    index=False
)

stuck_validation_df.to_csv(
    DEGRADATION_DIR
    / "fd001_stuck_validation.csv",
    index=False
)

print("All corruption-validation tables saved.")

In [ ]:
corruption_summary_df = pd.DataFrame([
    {
        "corruption": "noise",
        "validated_scenarios": len(noise_validation_df)
    },
    {
        "corruption": "bias",
        "validated_scenarios": len(bias_validation_df)
    },
    {
        "corruption": "drift",
        "validated_scenarios": len(drift_validation_df)
    },
    {
        "corruption": "dropout",
        "validated_scenarios": len(dropout_validation_df)
    },
    {
        "corruption": "stuck",
        "validated_scenarios": len(stuck_validation_df)
    }
])

corruption_summary_df.loc[
    len(corruption_summary_df)
] = {
    "corruption": "TOTAL",
    "validated_scenarios":
        corruption_summary_df[
            "validated_scenarios"
        ].sum()
}

display(corruption_summary_df)

corruption_summary_df.to_csv(
    DEGRADATION_DIR
    / "fd001_corruption_validation_summary.csv",
    index=False
)

In [ ]:
val_raw_check = (
    fd001_raw[
        fd001_raw["unit_id"].isin(
            val_engine_ids
        )
    ]
    .sort_values(
        ["unit_id", "cycle"]
    )
    .reset_index(drop=True)
)

val_raw_check = add_rul_targets(
    val_raw_check
)

assert list(
    val_raw_check.columns
) == list(
    val_raw.columns
)

for col in val_raw.columns:

    assert np.allclose(
        val_raw_check[col].to_numpy(),
        val_raw[col].to_numpy(),
        equal_nan=True
    )

print(
    "Final immutability check passed: "
    "clean validation data remain unchanged."
)